# 11 — TorchFX, torch.func, and Custom Ops/Extensions

Goal: graph transforms, advanced differentiation, and extending PyTorch.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Torch FX tracing

In [ ]:

import torch
import torch.nn as nn
from torch.fx import symbolic_trace

class FXModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(4, 8)
        self.l2 = nn.Linear(8, 2)
    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))

gm = symbolic_trace(FXModel())
print(gm.graph)

## 2. torch.func (grad/vmap concept)

Useful for per-sample gradients, Jacobians, vectorization.

In [ ]:

from torch import func

def f(params, x):
    W, b = params
    return (x @ W + b).sum()

W = torch.randn(5, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
x = torch.randn(10, 5)

g = func.grad(f)((W, b), x)
print("dW:", g[0].shape, "db:", g[1].shape)

## 3. C++ extension template (overview)

In [ ]:

cpp_ext_template = r'''
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CppExtension

setup(
    name="my_ext",
    ext_modules=[CppExtension("my_ext", ["my_ext.cpp"])],
    cmdclass={"build_ext": BuildExtension},
)
'''
print(cpp_ext_template)